# 🕸️ GraphRAG & Vector RAG — Streamlit Chatbot Application

This notebook manages and launches the **Streamlit Web Application** (`app.py`) built on top of your shared **GraphRAG** knowledge base and **ChromaDB** vector database.

---
### 🏗️ Architecture Overview
```
                 +---------------------------------------------+
                 |             Streamlit Chatbot UI            |
                 |  (Global / Local / DRIFT / Vector / Compare)|
                 +----------------------+----------------------+
                                        |
               +------------------------+------------------------+
               |                                                 |
               v                                                 v
  +--------------------------+                     +---------------------------+
  |     GraphRAG Tables      |                     |    ChromaDB Vector Store  |
  | (./ragtest/output/*.parquet) |                 |   (./notebook/chromadb)   |
  | - Entities & Relations   |                     | - 72 Embedded Chunks      |
  | - Leiden Communities     |                     | - Dense Semantic Search   |
  | - Community Summaries    |                     +---------------------------+
  +--------------------------+
```

---
### 🌟 Core Capabilities of the App:
1. **💬 Multi-Mode AI Chatbot**:
   - **GraphRAG Global Search**: High-level synthesis across hierarchical community reports.
   - **GraphRAG Local Search**: Entity neighborhood & relationship traversal.
   - **GraphRAG DRIFT Search**: Hybrid multi-hop search combining macro community context + granular entity edges.
   - **ChromaDB Vector RAG**: Standard dense semantic search over text chunks.
   - **Side-by-Side Comparison**: Run GraphRAG and Vector RAG in parallel to compare grounded reasoning.
2. **🔍 Full Grounding Auditability**: Expandable panels showing exact entities, edges, and chunks retrieved.
3. **🕸️ Interactive Knowledge Graph Tab**: Embedded draggable & zoomable PyVis physics graph.
4. **📊 Parquet & ChromaDB Explorers**: Live tables and semantic search tester for datasets.

## 1. Environment & Database Verification
Verify that our shared databases, Parquet files, and OpenAI credentials are ready.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import truststore
truststore.inject_into_ssl()
from dotenv import load_dotenv
import chromadb

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY', '')
masked_key = (api_key[:7] + '...') if api_key else 'None'
print(f'🔑 OpenAI API Key detected: {masked_key}')

# Verify Parquet Tables
print('\n📊 GraphRAG Tables in ./ragtest/output:')
for name in ['entities', 'relationships', 'nodes', 'community_reports']:
    path = Path(f'./ragtest/output/create_final_{name}.parquet')
    if path.exists():
        df = pd.read_parquet(path)
        print(f'  - create_final_{name}.parquet: {len(df)} rows')
    else:
        print(f'  ❌ Missing {path}')

# Verify ChromaDB
chroma_client = chromadb.PersistentClient(path='./notebook/chromadb')
collection = chroma_client.get_collection('paper_collection')
print(f'\n🗄️ ChromaDB collection "paper_collection": {collection.count()} chunks')

## 2. Programmatic Test of Shared Database Queries
Test querying both the GraphRAG tables and ChromaDB vector store before starting the web UI.

In [ ]:
from langchain_openai import ChatOpenAI
from app import query_graphrag_drift, query_chroma_rag, load_graph_data, load_chroma_db

entities_df, relationships_df, nodes_df, community_df = load_graph_data()
client, collection = load_chroma_db()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.2)

test_question = 'How does a company choose between RAG, fine-tuning, and PEFT approaches?'

print(f'❓ Question: {test_question}\n')
print('--- 🌐 GraphRAG DRIFT Result ---')
graph_ans, _ = query_graphrag_drift(test_question, community_df, relationships_df, llm)
print(graph_ans[:400] + '...\n')

print('--- 📚 ChromaDB Vector RAG Result ---')
vector_ans, _ = query_chroma_rag(test_question, collection, llm, num_results=3)
print(vector_ans[:400] + '...')

## 3. Launch the Streamlit Chatbot Application
Run the cell below to launch Streamlit in the background and obtain a local browser URL.

In [ ]:
import sys
import time
import subprocess
import urllib.request
from IPython.display import display, HTML

PORT = 8501
URL = f'http://localhost:{PORT}'

def is_running(url):
    try:
        with urllib.request.urlopen(f'{url}/_stcore/health', timeout=2) as response:
            return response.status == 200
    except Exception:
        return False

if is_running(URL):
    print(f'✅ Streamlit is already running at {URL}')
else:
    print(f'🚀 Starting Streamlit application on port {PORT}...')
    proc = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'app.py', '--server.port', str(PORT), '--server.headless', 'true'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    
    for _ in range(10):
        time.sleep(1)
        if is_running(URL):
            print('🎉 Streamlit is now LIVE!')
            break
    else:
        print('⚠️ Server started in background. Check URL below.')

display(HTML(f'''
<div style="background: linear-gradient(135deg, #1e1e2f, #2d2d48); padding: 20px; border-radius: 12px; border: 1px solid #4D96FF; margin: 15px 0;">
    <h3 style="color: #4D96FF; margin-top: 0;">🕸️ GraphRAG & Vector RAG Streamlit App</h3>
    <p style="color: #e2e8f0; font-size: 1.05rem;">The interactive web application is ready. Click the link below to open in your browser:</p>
    <a href="{URL}" target="_blank" style="display: inline-block; background: #4D96FF; color: white; padding: 10px 20px; border-radius: 8px; font-weight: bold; text-decoration: none;">👉 Open Streamlit App ({URL})</a>
</div>
'''))

## 4. Stop / Restart Streamlit App
If you need to stop or restart the running Streamlit server, execute this cell.

In [ ]:
import subprocess
result = subprocess.run(['pkill', '-f', 'streamlit run app.py'], capture_output=True, text=True)
print('🛑 Stopped running Streamlit instances. Re-run Cell 3 to restart.')